# 🚀 Treinamento e Testes do TRU-Net no Google Colab

Este notebook permite executar o treinamento do modelo cognitivo **TRU-Net** (RNN com módulos de viscosidade e comunicação limitada) utilizando o processamento de GPU gratuito do **Google Colab** (NVIDIA T4 / V100 / A100).

---
### 📌 Como usar:
1. No menu superior do Colab, vá em **Ambiente de execução (Runtime)** → **Alterar tipo de ambiente de execução (Change runtime type)** e selecione **T4 GPU**.
2. Execute as células sequencialmente abaixo.

## 1. Verificar GPU Disponível

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__)
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Dispositivo:', torch.cuda.get_device_name(0))
else:
    print('ATENÇÃO: GPU não ativada. Ative em Runtime -> Change runtime type -> T4 GPU')

## 2. Carregar os Arquivos do Projeto no Colab

Escolha uma das opções abaixo para carregar os arquivos no ambiente do Colab:
- **Opção A:** Clonar do GitHub (se o seu repositório estiver no GitHub)
- **Opção B:** Conectar ao Google Drive
- **Opção C:** Upload direto dos arquivos pelo navegador

In [ ]:
# === OPÇÃO A: Clone do GitHub ===
# !git clone https://github.com/SEU_USUARIO/TRU.git
# %cd TRU

# === OPÇÃO B: Conectar ao Google Drive ===
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/TRU

# === OPÇÃO C: Upload direto dos arquivos pelo navegador ===
from google.colab import files
import os

print('Diretório atual:', os.getcwd())
print('Arquivos presentes:', os.listdir('.'))

## 3. Instalar Dependências Necessárias

In [ ]:
!pip install -q pymupdf pandas

## 4. Executar Treinamento no Dataset de Português

Executa o script `train_portugues.py` com aceleração por GPU (`--device cuda`).

In [ ]:
!python train_portugues.py --data dataset_portugues_concurso.jsonl --epochs 10 --chunk_size 4 --vocab 2500 --batch 32 --device cuda

## 5. Teste de Inferência e Geração com o Modelo Treinado

In [ ]:
import torch
from tru_net.chunk_tokenizer import ChunkTokenizer
from tru_net.tru_net import TRUNet, TRUConfig

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tok = ChunkTokenizer.load('chunk_tokenizer.json')

cfg = TRUConfig(R=(16, 24, 12), D=(32, 32, 48), obs_dim=(32, 32, 32),
                msg_dim=8, global_dim=64, vocab_size=len(tok),
                emb_dim=64, n_actions=8)
model = TRUNet(cfg).to(device)
model.load_state_dict(torch.load('tru_net_texto.pt', map_location=device))
model.eval()

prompt = 'A crase e obrigatoria'
prefix_ids = torch.tensor(tok.encode(prompt), device=device)
out_ids = model.generate(prefix_ids, steps=20, think_steps=2)
gerado = tok.decode(out_ids)

print('Prompt:', prompt)
print('Gerado:', gerado)

## 6. Baixar Pesos e Tokenizer Treinados para o Computador

In [ ]:
from google.colab import files
files.download('tru_net_texto.pt')
files.download('chunk_tokenizer.json')
print('Download dos arquivos disparado com sucesso!')